# 

# 投影机内参标定与相机和投影机之间的外参标定


## 步骤一：生成投影机投影出的棋盘格

In [1]:
import cv2
import numpy as np
import os

# 参数配置
CALIB_IMG_DIR = "Data/Imgs/"          # 标定图像文件夹路径
SQUARE_SIZE = 10.0                    # 物理棋盘格方格边长 (mm，必须精确测量)
CHESSBOARD_CORNERS = (9, 7)           # 棋盘格内角点数量（列×行，与投影图案一致）
PROJ_RES = (1920, 1080)               # 投影机实际分辨率
CELL_SIZE_PX = 100                    # 投影机投影图案的方格像素宽度（与生成的投影图一致）

PROJ_PATTERN_PATH = "Data/Images/chessboard_proj.png"

# 计算棋盘格整体尺寸（外部黑白块数比角点数多1）
num_cols = CHESSBOARD_CORNERS[0] + 1
num_rows = CHESSBOARD_CORNERS[1] + 1
pattern_width = num_cols * CELL_SIZE_PX
pattern_height = num_rows * CELL_SIZE_PX

# 创建黑白交替棋盘格
chessboard = np.zeros((pattern_height, pattern_width), dtype=np.uint8)
for i in range(num_rows):
    for j in range(num_cols):
        # 偶数块涂白
        if (i + j) % 2 == 0:
            cv2.rectangle(
                chessboard,
                (j * CELL_SIZE_PX, i * CELL_SIZE_PX),
                ((j + 1) * CELL_SIZE_PX, (i + 1) * CELL_SIZE_PX),
                255,
                -1
            )

# 将棋盘格放到投影机分辨率中央
proj_pattern = np.zeros(PROJ_RES[::-1], dtype=np.uint8)
y_offset = (PROJ_RES[1] - pattern_height) // 2
x_offset = (PROJ_RES[0] - pattern_width) // 2
proj_pattern[y_offset:y_offset + pattern_height, x_offset:x_offset + pattern_width] = chessboard

# 保存
os.makedirs(os.path.dirname(PROJ_PATTERN_PATH), exist_ok=True)
cv2.imwrite(PROJ_PATTERN_PATH, proj_pattern)
print(f"投影机棋盘格图案已保存至：{PROJ_PATTERN_PATH}")

投影机棋盘格图案已保存至：Data/Images/chessboard_proj.png


## 步骤二：拍摄图像
将投影机中的棋盘投影到标定板上，仿照单目相机标定，从多个角度多个位置拍摄照片并保存，在每个拍摄角度下，让物理标定板的角点，和投影机投射图案的角点一一重合对齐，拍摄中必须保持投影机与相机的相对位置不变



### 具体操作（核心：1组姿态=2张照片，一一对应）
拍摄逻辑：先摆好一个标定板姿态 → 拍“仅标定板”照片（用于相机内参）→ 保持姿态不变 → 投射棋盘格 → 拍“投射+标定板”照片（用于外参+投影机内参）→ 更换姿态重复上述步骤。

#### 步骤1：确定拍摄姿态数量与多样性
- 总数量：至少15~25组姿态（姿态越多，标定精度越高，低于10组易导致参数漂移）；
- 姿态要求：必须覆盖「位置、角度、距离」的多样性，避免重复姿态，具体分类如下：
  | 姿态类型       | 操作说明                                                                 |
  |----------------|------------------------------------------------------------------------------------------|
  | 平移位置       | 标定板在相机视场中：左、右、上、下、中心5个位置；前后距离：近（相机1m内）、中（1~2m）、远（2~3m）3个距离 |
  | 倾斜角度       | 绕水平轴（上下倾斜，比如标定板上沿靠近相机、下沿远离）、绕垂直轴（左右倾斜）、绕垂直于板面的轴（旋转，比如标定板顺时针转30°），每个轴至少2个角度（±15°、±30°） |
  | 视场覆盖       | 至少2组姿态让标定板靠近相机视场边缘（比如左上角、右下角），确保覆盖相机全视场             |
  | 特殊姿态       | 1~2组轻微倾斜+平移的组合姿态（比如标定板左下方倾斜+靠近相机），模拟实际使用场景           |

#### 步骤2：单组姿态的拍摄细节
1. **摆放标定板**：
   - 用夹具固定标定板，确保板面平整（无翘曲），无遮挡（手部、夹具不挡住棋盘格角点）；
   - 确认标定板在「相机视场」和「投影机投射范围」内（相机能完整拍到标定板，投射棋盘格能完全覆盖标定板）。

2. **拍摄“仅标定板”照片（第1张）**：
   - 关闭投影机（或遮挡投射光线），仅让辅助灯照亮标定板；
   - 相机对准标定板，预览放大检查：所有内角点清晰、无模糊、无反光、无裁剪（标定板边缘不超出画面）；
   - 拍照并保存，照片命名为「calib_board_01.jpg」（01为姿态序号，后续依次递增）。

3. **拍摄“投射+标定板”照片（第2张，关键：姿态不变！）**：
   - 不移动标定板、相机、投影机（哪怕轻微动1mm都会导致角点不对应）；
   - 打开投影机，投射制作好的棋盘格，调整投影机角度（仅微调投射方向，不移动投影机位置），确保投射棋盘格完全覆盖实际标定板，且投射的方格与实际标定板的方格「无严重重叠遮挡」（比如投射棋盘格的角点和实际标定板的角点错开一点，方便算法区分；若重叠，可轻微旋转投影机投射方向，或微调标定板姿态，但需重新拍第1张“仅标定板”照片）；
   - 预览检查：投射的黑白方格对比度高（无发灰），实际标定板的角点仍清晰可见，无反光淹没；
   - 拍照并保存，照片命名为「proj_chessboard_01.jpg」（序号与“仅标定板”照片一致，比如01对应01）。

4. **确认照片合格**：
   - 放大两张照片，检查：
     - 「仅标定板」：内角点锐利，无模糊、无暗角、无裁剪；
     - 「投射+标定板」：投射棋盘格的内角点清晰（黑白交界分明），实际标定板的内角点未被投射图案遮挡，无环境光干扰导致的发灰。
   - 不合格则重拍（比如投射图案模糊，调整投影机焦距；实际标定板角点暗，调亮辅助灯）。

#### 步骤3：更换姿态重复拍摄
- 每组姿态拍摄完成后，松开夹具，调整标定板的位置/角度/距离（按步骤1的姿态要求），重复步骤2的“摆板→拍仅标定板→拍投射+标定板→校验”流程；
- 序号依次递增（比如第2组：calib_board_02.jpg + proj_chessboard_02.jpg），避免命名混乱；
- 拍摄过程中，若不小心移动了相机/投影机，即改变两者的相对位置，需全部重拍（外参标定依赖两者相对位置固定）。




## 步骤三：提取相机图像角点与投影像素对应

In [ ]:
import cv2
import numpy as np
import os
import pickle

# 参数配置

# 已知相机内参
CAM_INTRINSIC = np.array([
    [3.66498106e+03, 0, 1.25107241e+03],
    [0, 3.66359428e+03, 9.39504971e+02],
    [0, 0, 1]
])
CAM_DIST_COEFF = np.array([-5.41559209e-01,  3.68100960e-02,  
                           3.26237844e-03, -1.79983624e-04, 7.36295946e-01])

# 相位提取+系数拟合，与标定无关，仅用于后续计算相位-投影机坐标映射系数
def get_corner_phase(img, cam_corners):
    """从纯棋盘格图中提取角点周围的梯度相位（作为相位代理）"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    # 用Sobel算子计算梯度（棋盘格边缘梯度可解相）
    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)  # x方向梯度
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)  # y方向梯度
    # 解算包裹相位（-π ~ π）
    wrapped_phi_x = np.arctan2(grad_y, grad_x)  # 对应水平相位（投影机x坐标）
    wrapped_phi_y = np.arctan2(grad_x, grad_y)  # 对应垂直相位（投影机y坐标）
    
    # 提取每个角点周围5x5区域的平均相位（提高稳定性）
    phi_x_list, phi_y_list = [], []
    for (u, v) in cam_corners.astype(int):
        # 角点周围区域（避免超出图像边界）
        y1, y2 = max(0, v-2), min(img.shape[0], v+3)
        x1, x2 = max(0, u-2), min(img.shape[1], u+3)
        # 取区域平均相位
        avg_phi_x = np.mean(wrapped_phi_x[y1:y2, x1:x2])
        avg_phi_y = np.mean(wrapped_phi_y[y1:y2, x1:x2])
        phi_x_list.append(avg_phi_x)
        phi_y_list.append(avg_phi_y)
    return np.array(phi_x_list), np.array(phi_y_list)

def fit_phase_coefficients(all_phi_x, all_phi_y, proj_pattern_points):
    """用所有标定图的角点相位+已知投影机坐标，拟合线性系数"""
    # 合并所有标定图的相位数据（多图数据拟合更鲁棒）
    phi_x_flat = np.concatenate(all_phi_x)
    phi_y_flat = np.concatenate(all_phi_y)
    # 合并对应的投影机坐标（重复次数=标定图数量）
    proj_x_flat = np.tile(proj_pattern_points[:, 0], len(all_phi_x))
    proj_y_flat = np.tile(proj_pattern_points[:, 1], len(all_phi_y))
    
    # 线性拟合：phi → 投影机坐标（u_p = a_x*phi_x + b_x；v_p = a_y*phi_y + b_y）
    a_x, b_x = np.polyfit(phi_x_flat, proj_x_flat, deg=1)
    a_y, b_y = np.polyfit(phi_y_flat, proj_y_flat, deg=1)
    
    # 验证拟合误差
    proj_x_pred = a_x * phi_x_flat + b_x
    proj_y_pred = a_y * phi_y_flat + b_y
    avg_x_error = np.mean(np.abs(proj_x_pred - proj_x_flat))
    avg_y_error = np.mean(np.abs(proj_y_pred - proj_y_flat))
    
    return (a_x, b_x), (a_y, b_y), avg_x_error, avg_y_error

# 生成物理棋盘格的世界坐标（Z=0平面，单位mm）
objp = np.zeros((CHESSBOARD_CORNERS[0]*CHESSBOARD_CORNERS[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_CORNERS[0], 0:CHESSBOARD_CORNERS[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE

# 生成投影机面板的设计坐标（投影机像素坐标系，单位px）
proj_pattern_points = np.zeros((CHESSBOARD_CORNERS[0]*CHESSBOARD_CORNERS[1], 2), np.float32)
proj_pattern_points[:, :2] = np.mgrid[0:CHESSBOARD_CORNERS[0], 0:CHESSBOARD_CORNERS[1]].T.reshape(-1, 2)
proj_pattern_points *= CELL_SIZE_PX  # 与投射图案的格子尺寸严格一致

# 提取角点 & 建立匹配
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
obj_points = []     # 世界坐标（N×M×3）
cam_img_points = [] # 相机像素坐标（N×M×2）
proj_img_points = []# 投影机像素坐标（N×M×2）
all_phi_x = []      # 所有图的角点水平相位（对应投影机x坐标）
all_phi_y = []      # 所有图的角点垂直相位（对应投影机y坐标）

print("开始提取角点...")
for img_idx, img_name in enumerate(os.listdir(CALIB_IMG_DIR)):
    img_path = os.path.join(CALIB_IMG_DIR, img_name)
    img = cv2.imread(img_path)
    if img is None:
        print(f"跳过 {img_name}：无法读取图像")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 使用 SB 角点检测
    ret_cam, cam_corners = cv2.findChessboardCornersSB(
        gray, CHESSBOARD_CORNERS,
        cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FILTER_QUADS + cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    if not ret_cam:
        print(f"跳过 {img_name}：未检测到棋盘角点")
        continue

    cam_corners_refined = cv2.cornerSubPix(gray, cam_corners, (11, 11), (-1, -1), criteria)
    cam_corners_refined = cam_corners_refined.reshape(-1, 2)  # 统一维度：(M, 2)

    # 提取该图角点的相位
    phi_x, phi_y = get_corner_phase(img, cam_corners_refined)
    all_phi_x.append(phi_x)
    all_phi_y.append(phi_y)
    
    # 保存有效对应关系
    obj_points.append(objp)
    cam_img_points.append(cam_corners_refined)
    proj_img_points.append(proj_pattern_points.copy())

    # 可视化验证
    vis = img.copy()
    cv2.drawChessboardCorners(vis, CHESSBOARD_CORNERS, cam_corners_refined.reshape(-1,1,2), ret_cam, color=(0,0,255))
    cv2.putText(vis, f"Image: {img_idx+1}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)
    cv2.imshow("Detected Corners (Red)", vis)
    cv2.waitKey(30)  # 快速浏览，按ESC可退出

cv2.destroyAllWindows()

# 输出角点提取统计结果
total_imgs = len(os.listdir(CALIB_IMG_DIR))
valid_imgs = len(obj_points)
print(f"\n角点提取完成！")
print(f"统计：共 {total_imgs} 幅图像，有效图像 {valid_imgs} 幅")
if valid_imgs < 15:
    print("警告：有效图像少于15幅，可能导致标定结果不鲁棒，建议补充采集！")




## 步骤4：求解参数


In [ ]:
# stereoCalibrate 标定
if valid_imgs == 0:
    print("无有效标定图像，无法进行标定！")
else:
    print("\n开始标定相机-投影机系统...")
    # 投影机初始内参（基于分辨率猜测，合理即可）
    proj_K_init = np.array([
        [PROJ_RES[0]/2, 0, PROJ_RES[0]/2],
        [0, PROJ_RES[1]/2, PROJ_RES[1]/2],
        [0, 0, 1]
    ], dtype=np.float64)

    # 标定 flags 配置（固定相机内参，优化投影机参数）
    flags = (cv2.CALIB_USE_INTRINSIC_GUESS +    # 使用初始内参猜测
             cv2.CALIB_FIX_INTRINSIC +          # 固定相机内参
             cv2.CALIB_FIX_PRINCIPAL_POINT +    # 固定投影机主点
             cv2.CALIB_ZERO_TANGENT_DIST)       # 假设投影机无切向畸变

    # 执行双目标定
    rms, K_cam, dist_cam, K_proj, dist_proj, R_cam2proj, T_cam2proj, E, F = cv2.stereoCalibrate(
        obj_points, cam_img_points, proj_img_points,
        CAM_INTRINSIC, CAM_DIST_COEFF,
        proj_K_init, np.zeros(5),  # 投影机初始畸变设为0
        PROJ_RES,           # 投影机分辨率
        flags=flags,
        criteria=(cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS, 100, 1e-6)
    )

    # 拟合相位→投影机坐标系数
    print("\n开始拟合相位系数...")
    horiz_coeff, vert_coeff, x_error, y_error = fit_phase_coefficients(
        all_phi_x, all_phi_y, proj_pattern_points
    )
    print(f"相位系数拟合误差：x方向 {x_error:.2f}px，y方向 {y_error:.2f}px")
    
    
    # 验证重投影误差
    print("\n重投影误差验证...")
    total_proj_error = 0.0  # 投影机重投影误差
    total_cam_error = 0.0   # 相机重投影误差

    for i in range(valid_imgs):
        # 验证1：世界点 → 投影机像素（用标定后的参数）
        proj_pts_reproj, _ = cv2.projectPoints(
            obj_points[i], R_cam2proj, T_cam2proj, K_proj, dist_proj
        )
        
        proj_pts_reproj = proj_pts_reproj.reshape(-1, 2)
        # 单幅图投影机误差
        proj_error = cv2.norm(proj_img_points[i], proj_pts_reproj, cv2.NORM_L2) / len(proj_pts_reproj)
        total_proj_error += proj_error

        # 验证2：世界点 → 相机像素（验证相机内参是否准确）
        cam_pts_reproj, _ = cv2.projectPoints(
            obj_points[i], np.zeros((3,1)), np.zeros((3,1)),
            K_cam, dist_cam
        )
        cam_pts_reproj = cam_pts_reproj.reshape(-1, 2)
        # 单幅图相机误差
        cam_error = cv2.norm(cam_img_points[i], cam_pts_reproj, cv2.NORM_L2) / len(cam_pts_reproj)
        total_cam_error += cam_error

    # 计算平均误差
    avg_proj_error = total_proj_error / valid_imgs
    avg_cam_error = total_cam_error / valid_imgs
    
    # 创建结果文件夹
    os.makedirs("calib_results", exist_ok=True)
    # 整理标定结果
    calib_data = {
        "K_c": K_cam,                  # 相机内参（3x3）
        "dist_c": dist_cam,            # 相机畸变系数（5x1）
        "K_p": K_proj,                 # 投影机内参（3x3）
        "R": R_cam2proj,               # 相机→投影机旋转矩阵（3x3）
        "t": T_cam2proj.reshape(3, 1), # 相机→投影机平移向量（3x1）
        "horiz_coeff": horiz_coeff,    # 水平相位→x系数 (a_x, b_x)
        "vert_coeff": vert_coeff       # 垂直相位→y系数 (a_y, b_y)
    }   
    
    # 保存为pkl文件
    with open("calib_results/calib_data.pkl", "wb") as f:
        pickle.dump(calib_data, f)

    
    # 输出最终结果
    print(f"标定配置：")
    print(f"   - 物理棋盘格边长：{SQUARE_SIZE} mm")
    print(f"   - 投影机分辨率：{PROJ_RES[0]}×{PROJ_RES[1]}")
    print(f"   - 有效标定图像：{valid_imgs} 幅")
    print("\n误差指标：")
    print(f"   - 双目标定RMS误差：{rms:.4f} 像素")
    print(f"   - 投影机平均重投影误差：{avg_proj_error:.4f} 像素")
    print(f"   - 相机平均重投影误差：{avg_cam_error:.4f} 像素")
    print("\n投影机内参 K_proj：")
    print(f"{K_proj.round(4)}")
    print(f"\n投影机畸变系数 dist_proj：")
    print(f"{dist_proj.ravel().round(6)}")
    print(f"\n相机→投影机 外参：")
    print(f"   旋转矩阵 R：\n{R_cam2proj.round(4)}")
    print(f"   平移向量 T（mm）：\n{T_cam2proj.ravel().round(4)}")
    print("\n结果保存路径：calib_results/calib_data.pkl")
